In [26]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

# .env файлыг унших
load_dotenv()

# 2. Gemini API key-г шалгах
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY олдсонгүй. .env файлаа шалгана уу.")

# 3. LLM-ийг Gemini-ээр солих
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0.1
)

#def run_reflection_loop():
    # Даалгавар
  #  task_prompt = """
   #     Create a Python function `is_leap_year` that:
   #      1. Accepts an integer `year`.
   #      2. Returns True if it is a leap year, False otherwise.
   #      3. CRITICAL: You MUST NOT use the `calendar` or `datetime` libraries. You must write the logic manually.
   #      4. Include a docstring.
   #      5. The logic must be exactly correct: A year is a leap year if it is divisible by 4,
   #         BUT years divisible by 100 are NOT leap years, UNLESS they are also divisible by 400.
   #      6. Ensure the function handles strings that look like numbers (e.g., "2024").
    #     7. Add a performance optimization to check the most likely case first.
   # """
def run_reflection_loop():
    print("AI-д өгөх даалгавраа оруулна уу (Дуусгахдаа шинэ мөрөнд 'SEND' гэж бичээд Enter дарна уу):")

    lines = []
    while True:
        line = input()
        if line.strip().upper() == 'SEND':
            break
        lines.append(line)

    task_prompt = "\n".join(lines)

    if not task_prompt.strip():
        print("Даалгавар хоосон байна!")
        return 
    max_iterations = 3
    current_code = ""
    message_history = [HumanMessage(content=task_prompt)]

    for i in range(max_iterations):
        print("\n" + "="*25 + f" REFLECTION LOOP: ITERATION {i + 1} " + "="*25)

        # STAGE 1: GENERATE / REFINE
        if i == 0:
            print("\n>>> STAGE 1: GENERATING initial code...")
            response = llm.invoke(message_history)
            current_code = response.content
        else:
            print("\n>>> STAGE 1: REFINING code based on previous critique...")
            message_history.append(HumanMessage(content="Please refine the code using the critiques provided."))
            response = llm.invoke(message_history)
            current_code = response.content

        print("\n--- Generated Code (v" + str(i + 1) + ") ---\n" + current_code)
        message_history.append(response)

        # STAGE 2: REFLECT
        print("\n>>> STAGE 2: REFLECTING on the generated code...")

        reflector_prompt = [
            SystemMessage(content="""
               You are an extremely strict, perfectionist Senior Software Engineer.
                     You HATE even the smallest issues. You MUST find at least one improvement 
                     unless the code is literally perfect in every way (Style, Logic, Performance, Security).
                     Even if the logic is correct, critique the naming conventions or comments.
                     Do not say 'CODE_IS_PERFECT' unless you have absolutely nothing left to complain about.
            """),
            HumanMessage(content=f"Original Task:\n{task_prompt}\n\nCode to Review:\n{current_code}")
        ]

        critique_response = llm.invoke(reflector_prompt)
        critique = critique_response.content

        # STAGE 3: STOPPING CONDITION
        if "CODE_IS_PERFECT" in critique:
            print("\n--- Critique ---\nNo further critiques found. The code is satisfactory.")
            break

        print("\n--- Critique ---\n" + critique)
        message_history.append(HumanMessage(content=f"Critique of the previous code:\n{critique}"))

if __name__ == "__main__":
    run_reflection_loop()

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


AI-д өгөх даалгавраа оруулна уу (Дуусгахдаа шинэ мөрөнд 'Send' гэж бичээд Enter дарна уу):


KeyboardInterrupt: Interrupted by user